# 1 - Introdução ao LCEL (LangChain Expression Language)

**LCEL** é a forma moderna de compor pipelines no LangChain usando o operador `|` (pipe).

A ideia é simples: cada componente recebe uma entrada, processa e passa o resultado para o próximo.
Exatamente como o pipe do terminal Linux (`cat arquivo.txt | grep 'erro' | wc -l`).

```
prompt | model | output_parser
  │         │         │
  │         │         └── converte AIMessage → str
  │         └──────────── chama a API do LLM
  └────────────────────── formata a pergunta
```

### Por que LCEL?

| Antigo (deprecado) | Moderno (LCEL) |
|--------------------|----------------|
| `LLMChain(llm=..., prompt=...)` | `prompt \| model` |
| `ConversationChain(...)` | `prompt \| model \| parser` |
| `SequentialChain(...)` | encadeamento direto com `\|` |

> **`LLMChain` foi removido** no LangChain 0.3+. O equivalente moderno é simplesmente `prompt | llm`.

### Pacotes necessários

```bash
pip install langchain-openai langchain-core python-dotenv
```

## 1. Configuração inicial

Carregamos as variáveis de ambiente (chave da OpenAI) e instanciamos o modelo.

> **Modelos disponíveis**: prefira `gpt-4o-mini` em vez de `gpt-3.5-turbo-0125` — mais capaz,
> custo similar e com suporte ativo. O `gpt-3.5-turbo-0125` está em fase de descontinuação.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# gpt-4o-mini: substituto moderno do gpt-3.5-turbo — mais capaz e mesmo custo
model = ChatOpenAI(model="gpt-4o-mini")

## 2. Criando um Prompt Template

Um `ChatPromptTemplate` define o formato da mensagem enviada ao modelo.
As chaves entre `{}` são variáveis que serão preenchidas em tempo de execução.

| Método | Quando usar |
|--------|-------------|
| `from_template(str)` | Uma única mensagem do tipo `human` |
| `from_messages(lista)` | Múltiplas mensagens (`system`, `human`, `ai`) |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# {assunto} será substituído pelo valor passado no .invoke()
prompt = ChatPromptTemplate.from_template("Crie uma frase sobre: {assunto}")

# Inspecionando a estrutura do prompt
prompt

## 3. Compondo a chain com `|`

O operador `|` conecta os componentes. Cada componente deve ser compatível com o próximo:
a saída de um deve ser o tipo de entrada esperado pelo seguinte.

Neste caso:
- `prompt` recebe `dict` → produz `ChatPromptValue`
- `model` recebe `ChatPromptValue` → produz `AIMessage`

In [ ]:
# Chain básica: prompt → model
chain = prompt | model

# .invoke() executa toda a chain de ponta a ponta
# Retorna um AIMessage (objeto completo com metadados)
resposta = chain.invoke({"assunto": "Python"})
resposta

## 4. Adicionando um Output Parser

A chain acima retorna um `AIMessage` — um objeto com metadados de uso de tokens, model name, etc.
Na maioria dos casos, só queremos o **texto da resposta**.

O `StrOutputParser` extrai apenas o conteúdo textual do `AIMessage`.

```
AIMessage(content='Python é...', response_metadata={...})
        │
        ▼  StrOutputParser
'Python é...'
```

> **Dica**: sempre adicione `StrOutputParser()` no final quando a chain é o passo final
> e você só precisa do texto. Isso facilita o uso da resposta diretamente.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Chain completa: prompt → model → parser
# Agora .invoke() retorna diretamente uma string
chain = prompt | model | StrOutputParser()

resposta = chain.invoke({"assunto": "Python"})
print(resposta)  # str direta, sem AIMessage

## 5. Executando cada passo manualmente

Para entender o que acontece internamente, podemos chamar `.invoke()` em cada componente separadamente.
Isso é útil para **depurar** a chain quando algo não sai como esperado.

In [ ]:
input_data = {"assunto": "Python"}

# Passo 1: o prompt formata a entrada como uma mensagem de chat
prompt_formatado = prompt.invoke(input_data)
print("Após o prompt:")
print(prompt_formatado)
print()

In [ ]:
# Passo 2: o model recebe o prompt formatado e retorna um AIMessage
ai_message = model.invoke(prompt_formatado)
print("Após o model:")
print(ai_message)
print()
print(f"Tokens usados: {ai_message.response_metadata['token_usage']}")

In [ ]:
# Passo 3: o parser extrai só o texto do AIMessage
texto_final = StrOutputParser().invoke(ai_message)
print("Após o parser:")
print(texto_final)

## 6. Chain com RAG — unindo Retriever e LLM

A aplicação mais comum do LCEL é construir um sistema de **RAG (Retrieval-Augmented Generation)**:
o modelo responde perguntas baseando-se em documentos reais, não apenas no seu treinamento.

O fluxo:
```
pergunta
    │
    ├──► retriever  ──► chunks relevantes do PDF  ──► {contexto}
    │                                                       │
    └──────────────────────────────────────────► {pergunta} │
                                                            ▼
                                                    prompt | model | parser
                                                            │
                                                      resposta final
```

> **`RunnableParallel`**: executa múltiplos runnables em paralelo e combina os resultados
> em um dicionário. Aqui usamos para buscar o contexto e repassar a pergunta simultaneamente.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Carregando e dividindo o PDF
caminhos = ["files/LLM.pdf"]
paginas = []
for caminho in caminhos:
    loader = PyPDFLoader(caminho)
    paginas.extend(loader.load())

recur_split = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
documents = recur_split.split_documents(paginas)

print(f"Total de chunks: {len(documents)}")

In [ ]:
# Criando o vector store e o retriever
# FAISS: rápido, in-memory, ideal para protótipos
embeddings_model = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings_model
)

# MMR: retorna resultados relevantes E diversos (evita chunks repetitivos)
retriever = vectorstore.as_retriever(search_type="mmr")

print("Vector store criado com sucesso!")

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template_str = """Responda as perguntas do usuário com base no contexto fornecido.

Contexto: {contexto}

Pergunta: {pergunta}"""

template = ChatPromptTemplate.from_template(template_str)

# RunnableParallel executa os dois runnables ao mesmo tempo:
# - "pergunta": repassa o input sem modificação (RunnablePassthrough)
# - "contexto": busca os chunks relevantes no vector store (retriever)
setup_retrieval = RunnableParallel(
    pergunta=RunnablePassthrough(),
    contexto=retriever
)

# Chain completa de RAG
chain = setup_retrieval | template | model | StrOutputParser()

In [ ]:
# Testando a chain de RAG
resposta = chain.invoke("O que é LLM?")
print(resposta)

In [ ]:
# Inspecionando os documentos que o retriever encontrou
docs = retriever.invoke("O que é o LLM?")

print(f"Chunks recuperados: {len(docs)}\n")
for i, doc in enumerate(docs, 1):
    print(f"[{i}] Página {doc.metadata.get('page', '?')}:")
    print(doc.page_content[:200])
    print()

## Resumo

| Componente | Tipo de entrada | Tipo de saída | Papel na chain |
|------------|----------------|---------------|----------------|
| `ChatPromptTemplate` | `dict` | `ChatPromptValue` | Formata a mensagem para o LLM |
| `ChatOpenAI` | `ChatPromptValue` | `AIMessage` | Chama a API e gera a resposta |
| `StrOutputParser()` | `AIMessage` | `str` | Extrai só o texto da resposta |
| `RunnablePassthrough()` | qualquer | mesmo valor | Repassa o input sem modificar |
| `RunnableParallel(...)` | qualquer | `dict` | Executa múltiplos runnables em paralelo |

**Padrão LCEL completo:**

```python
# Chain simples
chain = prompt | model | StrOutputParser()

# Chain com RAG
chain = RunnableParallel(pergunta=RunnablePassthrough(), contexto=retriever) \
      | prompt | model | StrOutputParser()

# Execução
resposta = chain.invoke({"assunto": "Python"})  # chain simples
resposta = chain.invoke("O que é LLM?")          # chain com RAG
```

> **Próximo passo**: usar `with_structured_output()` para forçar o modelo a retornar
> dados estruturados (JSON, Pydantic) em vez de texto livre — ideal para tagging e extração.